We import all required modules

In [3]:
import os
import shutil
import tarfile
import glob
from sklearn.model_selection import train_test_split

from gcnn.paths import (
    DATA_DIR, ORIGINAL_DATASET, DATASET,
    TRAINING_SET, VALIDATION_SET, TEST_SET,
    ensure_data_dirs,
)


We define the required paths

In [4]:
# All paths are resolved automatically by gcnn.paths
# Override with GCNN_DATA_DIR env var if needed

ensure_data_dirs()

print(f"Data directory: {DATA_DIR}")


Data directory: C:\Git\gcnn\data


We download the database in our directory

In [ ]:
tar_path = ORIGINAL_DATASET / "dsgdb9nsd.xyz.tar"

# Only extract if not already done
if not any(DATASET.glob("dsgdb9nsd_*.xyz")):
    with tarfile.open(tar_path) as my_tar:
        my_tar.extractall(DATASET, filter="data")
    print("Extraction complete.")
else:
    print("Dataset already extracted, skipping.")


In [ ]:
files = glob.glob(str(DATASET / "dsgdb9nsd_*.xyz"))

print(f"Total number of entries: {len(files)}")


For this proof-of-principle calculation we are going to work only with 5% of the original dataset (i.e., instead of the original 140K, we will work with about 7K samples)

In [ ]:
# Optional: use a subset for quick experiments
# smaller_dataset, _ = train_test_split(files, test_size=0.95, random_state=42)
# files = smaller_dataset
# print(f"Using smaller dataset: {len(files)} entries")


We now split the smaller database into train (80%), validate(10%) and test (10%) sets, and store them in directories

In [ ]:
reminder_set, test = train_test_split(files, test_size=0.1, random_state=42)
train, validate = train_test_split(reminder_set, test_size=0.1, random_state=42)

print(f"test_size = {len(test)}")
print(f"validate_size = {len(validate)}")
print(f"train_size = {len(train)}")

total = len(test) + len(validate) + len(train)
print(f"total_size = {total}")
assert total == len(files), "Split sizes do not add up!"


Now, we just move the right files to the corresponding directories for the smaller size proof-of-concept

In [ ]:
# Only move if destination directories are empty
if not any(TEST_SET.glob("*.xyz")):
    for file in test:
        shutil.move(file, str(TEST_SET))
    for file in validate:
        shutil.move(file, str(VALIDATION_SET))
    for file in train:
        shutil.move(file, str(TRAINING_SET))
    print("Files moved to train/validation/test directories.")
else:
    print("Splits already exist, skipping move.")
